## Colab setup

Install dependencies and download the case data from the public repo (`github.com/jmmauricio/benchmarks_public`).

In [ ]:
%pip install --quiet pydae-core pydae-bps

In [ ]:
import urllib.request

RAW = 'https://raw.githubusercontent.com/jmmauricio/benchmarks_public/main/nts/cases/base'
for name in ('nts_base.hjson', 'nts_base_xy_0.json'):
    urllib.request.urlretrieve(f'{RAW}/{name}', name)

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt

from pydae import ssa
from pydae.bps import BpsBuilder
from pydae.core.builder import CasadiBuilder
from pydae.core.model import CasadiModel
from pydae.bps.utils.reporter import report_all
from pydae.bps.utils.validator import validate_all
from pydae.bps.lines import change_line
from pydae.utils import read_data

DATA = 'nts_base.hjson'
XY_0 = 'nts_base_xy_0.json'

grid = BpsBuilder(DATA, use_casadi=True)
grid.construct('nts_base')
bld = CasadiBuilder(grid.sys_dict).build()

In [ ]:
model = CasadiModel(bld)
model.decimation = 10
change_line(model, {"bus_j": "2", "bus_k": "3",
                    "X_pu": 0.6, "R_pu": 0.0, "Bs_pu": 0.0, "S_mva": 100})

model.ini({}, XY_0)

report_all(model, DATA)
validate_all(model, DATA)

In [ ]:
model.A_eval()
ssa.damp(model.A, model=model, sort='damp')
ssa.eig(model)
ssa.get_mode(model, f_min=0.1, f_max=0.5)